## 1. Configuration initiale et Importations

In [ ]:
# BLOC CODE 1
# --- Import des bibliothèques Python nécessaires au projet ---
import os
import requests
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
import numpy as np
from tqdm.notebook import tqdm
import base64
import io
from dotenv import load_dotenv
import time
import pandas as pd
from pathlib import Path
import re

In [ ]:
# BLOC CODE 2
# Charge le .env placé dans le dossier parent de "notebooks"
load_dotenv(os.path.join(os.path.dirname(os.getcwd()), ".env"))

# Utilisation de la variable d'env directement
HF_TOKEN = os.getenv("HUGGINGFACE_HUB_TOKEN") or os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Token Hugging Face manquant. Ajoute-le dans .env (HUGGINGFACE_HUB_TOKEN=...)")


### Variables de configuration

In [ ]:
# BLOC CODE 3
# Définition des variables de configuration
image_dir = "top_influenceurs_2024/IMG"

# Vérification de l’éxistence du dossier d’images
if not os.path.exists(image_dir):
  os.makedirs(image_dir)
  print(f"Dossier '{image_dir}' créé. Veuillez y ajouter des images .jpg ou .png.")
else:
  print(f"Dossier '{image_dir}' existant.")

max_images = 3
api_token = HF_TOKEN

## 2. Connexion à l’API d’Hugging Face

In [ ]:
# BLOC CODE 4
# Fonction d’extraction du numéro d’image avec son chemin
def extract_image_number(p: str) -> int:
    """
    Extrait le numéro dans le nom d'un fichier d'image avec son chemin, afin d'effectuer un tri numérique
    Exemple : 'top_influenceurs_2024/IMG/image_12.png' -> 12
    """
    name = Path(p).name
    m = re.findall(r"\d+", name)
    return int(m[-1]) if m else -1

In [ ]:
# BLOC CODE 5
API_URL = "https://router.huggingface.co/hf-inference/models/sayeed99/segformer_b3_clothes"
headers = {
    "Authorization": f"Bearer {api_token}"
}

# Lister les chemins des images à traiter
image_paths = []
image_extensions = [".png", ".jpeg", ".jpg"]

for filename in os.listdir(image_dir):
  if filename.lower().endswith(tuple(image_extensions)):
    image_paths.append(str(Path(image_dir) / filename))

image_paths.sort(key=extract_image_number) # Tri selon l’ordre des numéros des images (pas alphabétique)

if not image_paths:
  print(f"Aucune image trouvée dans '{image_dir}'. Veuillez ajouter des images.")
else:
  print(f"{len(image_paths)} image(s) à traiter : {image_paths}")

## 3. Fonctions Utilitaires pour le Traitement des Masques
Fonctions présentes :
1. `CLASS_MAPPING` : Un dictionnaire qui associe les noms de classes (ex: "Hat") à des identifiants numériques.
2. `COLOR_LIST` : Une liste de couleurs fixes associées à chaque classe.
3. Création des patches colorés pour la légende des masques.
4. `get_image_dimensions` : Récupérer les dimensions d'une image.
5. `decode_base64_mask` : Décoder un masque de base64 en une image (tableau NumPy) et le redimensionner.
6. `create_masks` : Combiner les masques de toutes les classes détectées en un seul masque de segmentation final, où chaque pixel a la valeur de l'ID de sa classe.

In [ ]:
# BLOC CODE 6
CLASS_MAPPING = {
    "Background": 0,
    "Hat": 1,
    "Hair": 2,
    "Sunglasses": 3,
    "Upper-clothes": 4,
    "Skirt": 5,
    "Pants": 6,
    "Dress": 7,
    "Belt": 8,
    "Left-shoe": 9,
    "Right-shoe": 10,
    "Face": 11,
    "Left-leg": 12,
    "Right-leg": 13,
    "Left-arm": 14,
    "Right-arm": 15,
    "Bag": 16,
    "Scarf": 17
}

# Liste de couleurs fixes pour chaque classe
COLOR_LIST = [
    "#000000",  # 0 Background (noir - on le rendra transparent dans l'overlay)
    "#ffff00",  # 1 / Chapeau
    "#ffa500",  # 2 / Cheveux
    "#ff00ff",  # 3 / Lunettes de soleil
    "#ff0000",  # 4 / Haut (vetement)
    "#00ffff",  # 5 / Jupe
    "#00ff00",  # 6 / Pantalon
    "#0000ff",  # 7 / Robe
    "#800080",  # 8 / Ceinture
    "#ffff00",  # 9 / Chaussure gauche
    "#008cff",  # 10 / Chaussure droite
    "#8cb4c7",  # 11 / Visage
    "#8cb4c7",  # 12 / Jambe gauche
    "#8cb4c7",  # 13 / Jambe droite
    "#8cb4c7",  # 14 / Bras gauche
    "#8cb4c7",  # 15 / Bras droit
    "#fe8000",  # 16 / Sac
    "#9213ff",  # 17 / Écharpe
]

# Création MANUELLE des handles (patchs colorés + label) pour la légende

legend_handles = []

# On veut créer (nom, id, couleur) → un Patch
classes_ordonnes = sorted(CLASS_MAPPING.items(), key=lambda kv: kv[1])  # tri par ID

for class_name, class_id in classes_ordonnes:
    color = COLOR_LIST[class_id]   # récupère la couleur correspondant à l'ID
    patch = Patch(facecolor=color, edgecolor='none', label=f"{class_id}: {class_name}")
    legend_handles.append(patch)

# Colormap catégorielle + normalisation discrète et pas continue
max_id = max(CLASS_MAPPING.values())
cmap = ListedColormap(COLOR_LIST[:max_id + 1])
norm = BoundaryNorm(range(max_id + 2), cmap.N)


def get_image_dimensions(img_path):
    """
    Get the dimensions of an image.

    Args:
        img_path (str): Path to the image.

    Returns:
        tuple: (width, height) of the image.
    """
    original_image = Image.open(img_path)
    return original_image.size

def decode_base64_mask(base64_string, width, height):
    """
    Decode a base64-encoded mask into a NumPy array.

    Args:
        base64_string (str): Base64-encoded mask.
        width (int): Target width.
        height (int): Target height.

    Returns:
        np.ndarray: Single-channel mask array.
    """
    mask_data = base64.b64decode(base64_string)
    mask_image = Image.open(io.BytesIO(mask_data))
    mask_array = np.array(mask_image)
    if len(mask_array.shape) == 3:
        mask_array = mask_array[:, :, 0]  # Prend le premier canal si RGB
    mask_image = Image.fromarray(mask_array).resize((width, height), Image.NEAREST)
    return np.array(mask_image)

def create_masks(results, width, height):
    """
    Combine multiple class masks into a single segmentation mask.

    Args:
        results (list): List of dictionaries with 'label' and 'mask' keys.
        width (int): Target width.
        height (int): Target height.

    Returns:
        np.ndarray: Combined segmentation mask with class indices.
    """
    combined_mask = np.zeros((height, width), dtype=np.uint8)  # Initialise avec le fond (0)

    # Traiter d'abord les masques non liés à l'arrière-plan
    for result in results:
        label = result['label']
        class_id = CLASS_MAPPING.get(label, 0)
        if class_id == 0:  # Ignorer l'arrière-plan
            continue
        mask_array = decode_base64_mask(result['mask'], width, height)
        combined_mask[mask_array > 0] = class_id

    for result in results:
        if result['label'] == 'Background':
            mask_array = decode_base64_mask(result['mask'], width, height)

    return combined_mask

## 4. Segmentation d'une Seule Image
Étapes :

1. Choisir une image.
2. Ouvrir l'image en mode binaire ("rb") et lire son contenu (data).
3. Déterminer le Content-Type (par exemple, "image/jpeg" ou "image/png").
4. Envoyer la requête POST à l'API avec requests.post() en passant l'URL, les headers et les données.
5. Vérifier le code de statut de la réponse. Une erreur sera levée si le code n'est pas 2xx (succès) grâce à response.raise_for_status().
6. Convertir la réponse JSON en un dictionnaire Python avec response.json().
7. Utiliser nos fonctions get_image_dimensions et create_masks pour obtenir le masque final.
8. Afficher l'image originale et le masque segmenté.

In [ ]:
# BLOC CODE 7
# Fonction d’appel API
def api_call(data_image: bytes):
    """
    Appelle l'API Hugging Face avec une image en binaire.

    Args:
        data_image : Image à envoyer à l'API
    
    Returns:
        Liste de dictionnaires
    """
    try:
        data_response = requests.post(
            API_URL,
            headers={"Content-Type": "image/png", **headers},
            data=data_image,
            timeout=15
        )
        data_response.raise_for_status() # Vérification qu'il y a aucune erreur HTTP
        return data_response.json()
    except requests.RequestException as err:
        print(f"[ERREUR API] {err}")
        return None

In [ ]:
# BLOC CODE 8
# Pour une seule image
if image_paths:
  single_image_path = image_paths[1]
  print(f"Traitement de l'image : {single_image_path}")

  try:
    with open(single_image_path, "rb") as f: # Ouverture du fichier en lecture binaire
      image_data = f.read()
    # --- Appel fonction API ---
    json_response = api_call(image_data)
    
    # --- Exploitation des résultats ---
    image_size = get_image_dimensions(single_image_path)
    masks = create_masks(json_response, image_size[0], image_size[1])
    original = Image.open(single_image_path).convert("RGB")
    # --- Affichage des résultats ---
    fig, axes = plt.subplots(1, 3, figsize=(12, 6)) # Création d’un graphique et de trois sous-graphiques

    # Affiche l'original
    axes[0].imshow(original)
    axes[0].set_title("Original")
    axes[0].axis("off")

    # Affiche le masque
    axes[1].imshow(masks, cmap=cmap, norm=norm)
    axes[1].set_title("Masques")
    axes[1].axis("off")

    # Emplacement de la légende
    axes[2].set_title("Légende")
    axes[2].axis("off")

    # Création de la légende des classes
    leg = axes[2].legend(
    handles=legend_handles,      # Liste de légendes (patch + label)
    loc="upper left",
    ncol=1,
    frameon=True,
    borderaxespad=0.5
    )

    # Affichage
    plt.tight_layout()
    plt.show() # Affichage les 3 images
    
  except Exception as e:
        print(f"Une erreur est survenue : {e}")
else:
    print("Aucune image à traiter. Vérifiez la configuration de 'image_dir' et 'max_images'.")


## 5. Segmentation de Plusieurs Images (Batch)
Maintenant que nous savons comment traiter une image, nous pouvons créer une fonction pour en traiter plusieurs. Cette fonction va boucler sur la liste `image_paths` et appliquer la logique de segmentation à chaque image. Nous utiliserons `tqdm` pour avoir une barre de progression.

In [ ]:
# BLOC CODE 9
def segment_images_batch(list_of_image_paths):
    """
    Segmente une liste d'images en utilisant l'API Hugging Face.

    Args:
        list_of_image_paths (list): Liste des chemins vers les images.

    Returns:
        list: Liste des masques de segmentation (tableaux NumPy).
              Contient None si une image n'a pas pu être traitée.
    """
    batch_segmentations = []

    for img in tqdm(list_of_image_paths, desc="Traitement des images", unit="img"):
        try:
            with open(img, "rb") as f:
                image_data = f.read()
            response = api_call(image_data)
            if not response:
                batch_segmentations.append(None)
                time.sleep(1)
                continue

            image_size = get_image_dimensions(img)
            mask = create_masks(response, image_size[0], image_size[1])
            batch_segmentations.append(mask)

        except Exception as e:
            print(f"[ERREUR] {img}: {e}")
            batch_segmentations.append(None)
        
        time.sleep(1) # Pause d’une seconde entre chaque appel API

    return batch_segmentations

# Appeler la fonction pour segmenter les images listées dans image_paths
if image_paths:
    print(f"\nTraitement de {len(image_paths)} image(s) en batch...")
    batch_seg_results = segment_images_batch(image_paths)
    print("Traitement en batch terminé.")
else:
    batch_seg_results = []
    print("Aucune image à traiter en batch.")

## 6. Affichage des Résultats en Batch
Nous allons maintenant créer une fonction pour afficher les images originales et leurs segmentations correspondantes côte à côte, dans une grille.

In [ ]:
# BLOC CODE 10
def display_segmented_images_batch(original_image_paths, segmentation_masks):
    """
    Affiche les images originales et leurs masques segmentés.

    Args:
        original_image_paths (list): Liste des chemins des images originales.
        segmentation_masks (list): Liste des masques segmentés (NumPy arrays).
    """
    # Création de graphiques et sous-graphiques
    global_fig, global_axes = plt.subplots(len(original_image_paths),3, figsize=(12, (6 * len(original_image_paths))), squeeze=False)

    for i, path in enumerate(original_image_paths):
        original_image = Image.open(path).convert("RGB")
        # Création de l’affichage de l’image originale
        global_axes[i, 0].imshow(original_image)
        global_axes[i, 0].set_title("Image " + str(i) + ": Originale")
        global_axes[i, 0].axis("off")

        # Création de l’affichage du masque
        global_axes[i, 1].imshow(segmentation_masks[i], cmap=cmap, norm=norm, interpolation="nearest")
        global_axes[i, 1].set_title("Masque")
        global_axes[i, 1].axis("off")

        # Emplacement de la légende
        global_axes[i, 2].set_title("Légende")
        global_axes[i, 2].axis("off")

        # Création de la légende des classes
        leg = global_axes[i, 2].legend(
            handles=legend_handles,
            loc="upper left",
            ncol=1,
            frameon=True,
            borderaxespad=0.5
        )
    
    # Affichage de tous les sous-graphiques dans un seul graphique
    plt.tight_layout()
    plt.show()

# Afficher les résultats du batch
if batch_seg_results:
    display_segmented_images_batch(image_paths, batch_seg_results)
else:
    print("Aucun résultat de segmentation à afficher.")

## Validation du modèle
Cette étape permet de valider un modèle en analysant sa performance avec des métriques sur chaque classe, puis au global.
La métrique principale retenue est IoU (Intersection over Union), plus adaptée à la segmentation, mais les métriques Dice et F1-score seront aussi calculées.

In [ ]:
# BLOC CODE 11
# ------ Configuration ------

mask_dir = "top_influenceurs_2024/Mask"

# Vérification de l’éxistence du dossier de masques
if not os.path.exists(mask_dir):
  os.makedirs(mask_dir)
  print(f"Dossier '{mask_dir}' créé. Veuillez y ajouter des masques .png.")
else:
  print(f"Dossier '{mask_dir}' existant.")

# Liste des chemins des masques à traiter
mask_paths = []

for maskname in os.listdir(mask_dir):
  if maskname.lower().endswith(".png"):
    mask_paths.append(str(Path(mask_dir) / maskname))

mask_paths.sort(key=extract_image_number) # Tri selon l’ordre des numéros des masques (non alphabétique)

if not mask_paths:
  print(f"Aucun masque trouvé dans '{mask_dir}'. Veuillez ajouter des masques.")
else:
  print(f"{len(mask_paths)} masque(s) à traiter : {mask_paths}")

In [ ]:
# BLOC CODE 12 — Normalisation des chemins & mapping image↔masque

from pathlib import Path

# 1) Normalise les séparateurs (évite le mélange \ /)
image_paths = [str(Path(p)) for p in image_paths]
mask_paths  = [str(Path(p)) for p in mask_paths]

# 2) Trie par numéro d’image (utilise la version robuste d'extract_image_number)
image_paths.sort(key=extract_image_number)
mask_paths.sort(key=extract_image_number)

# 3) Associe image ↔ masque par numéro (plus sûr que l'index de liste)
mask_by_num = { extract_image_number(p): p for p in mask_paths }

# 4) Logs de contrôle (format POSIX pour lisibilité)
print("[OK] Chemins normalisés et mappés.")
if image_paths:
    print("[DEBUG] image_paths[0]:", Path(image_paths[0]).as_posix())
if mask_paths:
    print("[DEBUG] mask_paths[0]:", Path(mask_paths[0]).as_posix())
print("[DEBUG] nb images:", len(image_paths), "| nb masques:", len(mask_paths))


In [ ]:
# BLOC CODE 13
# ------ Définition des fonctions ------

# --- Fonction d’ouverture de masque réel en PNG et conversion en array ---
def load_png_mask(mask_path, width=None, height=None):
    """
    Chage le masque de segmentation réel à partir d'un fichier PNG local et le convertit en array Numpy.

    Args:
        mask_path (str): Chemin du fichier masque PNG.
        width (int, optionel): Largeur cible. Si définit à None, garde la largeur originale.
        height (int, optionel): Hauteur cible. Si définit à None, garde la hauteur originale.

    Returns:
        np.ndarray: Array à un seul canal (Identifiant de classe par pixel).
    """
    mask_image = Image.open(mask_path)
    mask_array = np.array(mask_image)

    # Si plusieurs cannaux (RGB), on garde le premier
    if len(mask_array.shape) == 3:
        mask_array = mask_array[:,:, 0]

    # Redimensionnement optionnel
    if width is not None and height is not None:
        mask_image = Image.fromarray(mask_array).resize((width, height), Image.NEAREST)
        mask_array = np.array(mask_image)
    
    return np.array(mask_array)

# --- Vérifications des arrays ---
def _validate_inputs(pred: np.ndarray, gt: np.ndarray):
    if pred.shape != gt.shape: # Si la prédiction et le réel ont des tailles différentes
        raise ValueError(f"Shapes mismatch: pred{pred.shape} vs gt{gt.shape}")
    if pred.ndim != 2 or gt.ndim != 2: # Si un des deux arrays est de dimension supérieure à 2
        raise ValueError(f"Attendu des masques 2D (H, W).")

# --- Comptage des True Positive, False Positive et des False Negative pour la classe k ---
def counts_for_class(pred: np.ndarray, gt: np.ndarray, k: int):
    """
    Retourne TP (True Positive), FP (False Positive), FN (False Negative) pour la classe k.
    """
    _validate_inputs(pred, gt)
    p = (pred == k)                                                 # p = Pixels prédits d’une classe
    g = (gt == k)                                                   # g = Pixels réels d’une classe
    tp = np.logical_and(p, g).sum(dtype=np.int64)                   # True Positive = Somme des pixels prédits et justes
    fp = np.logical_and(p, np.logical_not(g)).sum(dtype=np.int64)   # False Positive = Somme des pixels prédits mais faux
    fn = np.logical_and(np.logical_not(p), g).sum(dtype=np.int64)   # False Negative = Somme des pixels non prédits mais justes
    inter = tp                                                      # Intersection = True Positive
    union = np.logical_or(p, g).sum(dtype=np.int64)                 # Union = Somme des pixels prédits et des pixels réels d’une classe
    pred_pixels = p.sum(dtype=np.int64)                             # Prediction Pixels = Somme des pixels prédits d’une classe
    gt_pixels = g.sum(dtype=np.int64)                               # Ground Truth Pixels = Somme des pixels réels d’une classe

    return int(tp), int(fp), int(fn), int(inter), int(union), int(pred_pixels), int(gt_pixels)

# --- Calcul de la métrique IoU pour la classe k ---
def iou_from_counts(tp: int, fp: int, fn: int):
    denominateur = tp + fp + fn
    if denominateur > 0:
        iou = tp / denominateur
        return iou
    else:
        return np.nan
    
# --- Calcul de la métrique DICE / F1 score pour la classe k ---
def dice_from_counts(tp: int, fp: int, fn: int):
    denominateur = 2 * tp + fp + fn
    if denominateur > 0:
        dice = (2 * tp) / denominateur
        return dice
    else:
        return np.nan
    
# --- Calcul de la précision pour la classe k ---
def precision_from_counts(tp: int, fp: int, fn: int):
    denominateur = tp + fp
    if denominateur > 0:
        precision = tp / denominateur
        return precision
    else:
        return np.nan
    
# --- calcul de métrique pour toutes les classes ---
def metric_per_class(pred: np.ndarray, gt: np.ndarray, classes, ignore_index=0, which="iou"):
    """
    Calcule une métrique par classe + une macro-moyenne.
    which ∈ {"iou", "dice", "precision"}

    Args:
        pred (array): Array du masque prédit.
        gt (array): Array du masque réel.
        classes (list): Liste des classes / catégories (de 0 à 17).
        ignore_index (int): Index de la classe à exclure (par défaut 0: le fond).
        which (str): Choix de la métrique à calculer.

    Returns:
        values (dict): Dictionnaire servant à stocker la valeur de la métrique choisie pour chaque classe et pour la moyenne.
    """
    _validate_inputs(pred, gt)

    values = {}
    pool = []

    for k in classes:
        if k == ignore_index:
            values[k] = np.nan
            continue
            
        tp, fp, fn, inter, union, pred_pixels, gt_pixels = counts_for_class(pred, gt, k)
        if which == "iou":
            val = iou_from_counts(tp, fp, fn)
        elif which == "dice":
            val = dice_from_counts(tp, fp, fn)
        elif which == "precision":
            val = precision_from_counts(tp, fp, fn)
        else:
            raise ValueError("which doit être 'iou', 'dice' ou 'precision'.")
        
        values[k] = val
        if not np.isnan(val):
            pool.append(val)
        
    # Moyenne de la métrique
    values[f"macro_{which}"] = float(np.mean(pool)) if pool else np.nan
    return values

# --- Calcul des données IoU pour le graphique en barres à partir des deux masques (prédit et réel) ---
def iou_bar_data(pred_mask: np.ndarray, real_mask: np.ndarray):
    """
    Calcul des données IoU pour le graphique en barres à partir des deux masques (prédit et réel).

    Args:
        pred_mask (array): Array du masque prédit.
        real_mask (array): Array du masque réel.

    Returns:
        ids (list): Liste des IDs des classes présentes dans l'image.
        vals (list): Liste des valeurs d'IoU des classes présentes dans l'image.
        colors (list): Liste des couleurs des classes présentes dans l'image.
        labels (list): Liste des noms des classes présentes dans l'image.
    """
    classes = list(range(18))
    iou = metric_per_class(pred_mask, real_mask, classes=classes, ignore_index=0, which="iou")

    # ID -> nom de classe
    ID_TO_NAME = {v: k for k, v in CLASS_MAPPING.items()}

    ids, vals, colors, labels = [], [], [], []
    for k in range(1, 18): # ignore le fond
        val = iou.get(k, np.nan) # On n’affiche que les classes présentes dans l’image
        if not np.isnan(val):
            ids.append(k)
            vals.append(val)
            colors.append(COLOR_LIST[k])
            labels.append(f"{k}\n{ID_TO_NAME.get(k, str(k))}")

    miou = iou.get("macro_iou", np.nan)
    return ids, vals, colors, labels, miou

# --- Traitement d’une image : Appel API, chargement masque réel, retourne tout le nécessaire ---
def process_one_image(image_path: str, mask_path: str):
    """
    Traitement d'une image : Appel API, chargement masque réel, retourne tout le nécessaire.

    Args:
        image_path (str): Nom du chemin de l'image originale.
        mask_path (str): Nom du chemin du masque réel.

    Returns:
        original (Image): Image originale.
        pred_mask (Array): Array du masque prédit.
        real_mask (Array): Array du masque réel.
        ids (list): Liste des IDs des classes présentes dans l'image.
        vals (list): Liste des valeurs d'IoU des classes présentes dans l'image.
        colors (list): Liste des couleurs des classes présentes dans l'image.
        labels (list): Liste des noms des classes présentes dans l'image.
    """
    # Image originale et prédiction
    with open(image_path, "rb") as f:
        image_data = f.read()

    json_response = api_call(image_data)
    w, h = get_image_dimensions(image_path)
    pred_mask = create_masks(json_response, w, h)
    original = Image.open(image_path).convert("RGB")

    # Masque réel aligné
    real_mask = load_png_mask(mask_path, width=w, height=h)

    # Données pour graphique en barre
    ids, vals, colors, labels, miou = iou_bar_data(pred_mask, real_mask)

    return original, pred_mask, real_mask, (ids, vals, colors, labels, miou)

### Évaluation d'une image
Comparaison Original + prédiction + réel + IoU (par classe et moyen) côte à côte sur une grille.

In [ ]:
# BLOC CODE 14
def plot_row_one(image_path: str, mask_path: str, figsize=(12, 5)):
    """
    Affichage côte à côte de : l'image originale, la prédiction du masque, le masque réel, le graphique en barre des score IoU des classes.

    Args:
        image_path (str): Nom du chemin de l'image originale.
        mask_path (str): Nom du chemin du masque réel.
        figsize : Dimensions de la figure. Par défaut: 12 pouces par 5 pouces.

    Returns:
        Affichage de l'image.
    """
    original, pred_mask, real_mask, barpack = process_one_image(image_path, mask_path)
    ids, vals, colors, labels, miou = barpack

    fig, axes = plt.subplots(1, 4, figsize=figsize)
    plt.subplots_adjust(wspace=0.05)

    # Affichage image originale
    axes[0].imshow(original)
    axes[0].set_title("Original")
    axes[0].axis("off")

    # Affichage masque prédit
    axes[1].imshow(pred_mask, cmap=cmap, norm=norm)
    axes[1].set_title("Prédiction masque")
    axes[1].axis("off")

    # Affichage masque réel
    axes[2].imshow(real_mask, cmap=cmap, norm=norm)
    axes[2].set_title("Masque réel")
    axes[2].axis("off")

    # Graphique en barre d’IoU
    if ids:
        x = np.arange(len(ids))
        axes[3].bar(x, vals, color=colors)

        for i, v in enumerate(vals):
            axes[3].text(x[i], v + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
        axes[3].set_xticks(x)
        axes[3].set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
        axes[3].set_ylim(0, 1)
        axes[3].set_ylabel("IoU")
        title_miou = f"mIoU={miou:.2f}" if np.isfinite(miou) else "mIoU=NA"
        axes[3].set_title(f"IoU par classe ({title_miou})")
        axes[3].grid(axis="y", linestyle="--", alpha=0.4)
    else:
        axes[3].text(0.5, 0.5, "Aucune classe (hors fond) détectée", ha="center",va="center")
        axes[3].axis("off")

    plt.tight_layout(pad=1.0, w_pad=0.2)
    plt.show()


### Affichage d’une image
Image originale + prédiction masque (avec le modèle) + masque réel + évaluation IoU par classe. (1 ligne x 4 colonnes)

In [ ]:
# BLOC CODE 15
img_idx = 9 # Choix de l’index de l’image à afficher

plot_row_one(image_paths[img_idx], mask_paths[img_idx])

### Évaluation de toutes les images
Comparaison Original + prédiction + réel + IoU (par classe et moyen) côte à côte sur une grille (N lignes x 4 colonnes).

In [ ]:
# BLOC CODE 16
def plot_all_rows(image_paths, mask_paths):
    assert len(image_paths) == len(mask_paths), "Listes images/masques de taille différentes"
    n = len(image_paths)
    if n == 0:
        print("Rien à afficher."); return
    
    fig, axes = plt.subplots(n, 4, figsize=(12, 5*n))

    if n == 1: # Si n == 1, axes est 1D : on homogénéise en liste
        axes = np.array([axes])
    
    plt.subplots_adjust(wspace=0.05, hspace=0.25)

    for row in tqdm(range(n), desc="Traitement des images", unit="img"):
        try:
            original, pred_mask, real_mask, barpack = process_one_image(image_paths[row], mask_paths[row])
            ids, vals, colors, labels, miou = barpack

            # Affichage image originale
            axes[row, 0].imshow(original)
            axes[row, 0].set_title("Image " + str(row) + ": Originale")
            axes[row, 0].axis("off")

            # Affichage prédiction masque
            axes[row, 1].imshow(pred_mask, cmap=cmap, norm=norm)
            axes[row, 1].set_title("Prédiction masque")
            axes[row, 1].axis("off")

            # Affichage masque réel
            axes[row, 2].imshow(real_mask, cmap=cmap, norm=norm)
            axes[row, 2].set_title("Masque réel")
            axes[row, 2].axis("off")

            # Graphique en barre d’IoU
            ax = axes[row, 3]
            if ids:
                x = np.arange(len(ids))
                ax.bar(x, vals, color=colors)
                for i, v in enumerate(vals):
                    ax.text(x[i], v + 0.02, f"{v:.2f}", ha="center", va="bottom", fontsize=8)
                ax.set_xticks(x)
                ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
                ax.set_ylim(0, 1)
                title_miou = f"mIoU={miou:.2f}" if np.isfinite(miou) else "mIoU=NA"
                ax.set_title(title_miou)
                ax.grid(axis="y", linestyle="--", alpha=0.4)
            else:
                ax.text(0.5, 0.5, "Aucune classe (hors fond) détectée", ha="center",va="center")
                ax.axis("off")

        except Exception as e: # En cas de problème sur une ligne
            for c in range(4):
                axes[row, c].axis("off")
            axes[row, c].text(0.5, 0.5, f"Erreur ligne {row} : {e}", ha="center", va="center", color="red")
    
    plt.tight_layout(pad=1.0, w_pad=0.2, h_pad=0.6)
    plt.show()

In [ ]:
# BLOC CODE 17
plot_all_rows(image_paths, mask_paths)

### Calculs des métriques
Récapitulatif de toutes les métriques pour toutes les images et toutes les classes.

In [ ]:
# BLOC CODE 18
# --- Construction d’un DataFrame pour une image ---

# ID -> Nom + Couleur
ID_TO_NAME = {v: k for k, v in CLASS_MAPPING.items()}

def evaluate_one_image_to_df(image_id: str, pred: np.ndarray, gt: np.ndarray, classes=range(18), ignore_index=0):
    """
    Retourne un DataFrame avec 1 ligne par classe (0..17) pour l'image donnée.
    - Laisse les métriques à NaN si union==0 (classe absente dans l'image)
    - Met aussi NaN pour la classe 'ignore_index' (par défaut 0 = Background)
    """
    rows = []
    for k in classes:
        tp, fp, fn, inter, union, pred_px, gt_px = counts_for_class(pred, gt, k)

        if k == ignore_index:
            iou = dice = prec = np.nan
        else:
            iou = iou_from_counts(tp, fp, fn) if union > 0 else np.nan
            dice = dice_from_counts(tp, fp, fn) if union > 0 else np.nan
            prec = precision_from_counts(tp, fp, fn) if (tp + fp) > 0 else np.nan

        rows.append({
            "image_id": image_id,
            "class_id": k,
            "class_name": ID_TO_NAME.get(k, str(k)),
            "color": COLOR_LIST[k],
            "tp": tp, "fp": fp, "fn": fn,
            "inter": inter, "union": union,
            "pred_pixels": pred_px, "gt_pixels": gt_px,
            "present": bool(union > 0 and k != ignore_index),
            "iou": iou, "dice": dice, "precision": prec,
        })
    return pd.DataFrame(rows)

In [ ]:
# BLOC CODE 19
img_idx = 9 # Choix de l’index de l’image à afficher

orig, pred_mask, real_mask, liste = process_one_image(Path(image_paths[img_idx]) ,Path(mask_paths[img_idx]))

# DataFrame pour une image
df_one = evaluate_one_image_to_df(
    image_id=Path(image_paths[img_idx]).name,
    pred=pred_mask,
    gt=real_mask,
    classes=range(18),
    ignore_index=0
)

df_one


In [ ]:
# BLOC CODE 20 — DF global (toutes les images)

# Associe image ↔ masque par numéro (sécurisé si les listes ne sont pas strictement alignées)
mask_by_num = { extract_image_number(p): p for p in mask_paths }

dfs = []
for img_path in tqdm(image_paths, desc="Traitement des images", unit="img"):
    num = extract_image_number(img_path)
    mask_path = mask_by_num.get(num)
    if mask_path is None:
        print(f"[WARN] Pas de masque pour l’image n°{num} ({img_path})")
        continue

    # Traite l’image
    original, pred_mask, real_mask, _ = process_one_image(Path(img_path), Path(mask_path))

    # Alimente le DF (1 ligne par classe)
    df_one = evaluate_one_image_to_df(
        image_id=Path(img_path).name,
        pred=pred_mask,
        gt=real_mask,
        classes=range(18),
        ignore_index=0
    )
    dfs.append(df_one)

df_all = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame()
print("[OK] Lignes dans df_all :", len(df_all))
df_all.head()
